# 07 — RAG Pipeline

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/07_rag_pipeline.ipynb)

## 📌 What is it?
Retrieval-Augmented Generation (RAG) combines a retrieval system with an LLM: retrieve relevant documents, then generate grounded responses based on them.

## 🧠 Why AI Engineers need this
RAG is the most common production AI architecture. It lets LLMs answer questions about private data, recent events, or large document stores without fine-tuning.

In [ ]:
# ── RAG ARCHITECTURE ──
print("""
RAG Pipeline — Full Architecture

  USER QUERY
      │
      ▼
┌─────────────┐     ┌──────────────────────────────────────┐
│   QUERY     │     │           VECTOR STORE                │
│  EMBEDDING  │────▶│  doc1_emb  doc2_emb  doc3_emb  ...   │
│  (API/local)│     │         (pre-indexed)                 │
└─────────────┘     └──────────────────┬───────────────────┘
                                        │
                              Top-K most similar docs
                                        │
                                        ▼
┌──────────────────────────────────────────────────────────┐
│                   PROMPT BUILDER                          │
│                                                           │
│  System: You are a helpful assistant.                     │
│  Context: [doc3] [doc1] [doc5]   ← retrieved chunks      │
│  User: {original query}                                   │
└────────────────────────────┬─────────────────────────────┘
                              │
                              ▼
                        ┌──────────┐
                        │   LLM    │
                        │ (GPT-4o/ │
                        │ Claude)  │
                        └────┬─────┘
                              │
                              ▼
                       GROUNDED ANSWER
                    (with source citations)
""")

In [ ]:
import numpy as np
import json
from typing import List, Dict, Optional

# ── COMPLETE RAG PIPELINE IMPLEMENTATION ──

class Document:
    """A document chunk with text and metadata."""
    def __init__(self, text: str, metadata: dict = None):
        self.text = text
        self.metadata = metadata or {}
        self.embedding: Optional[np.ndarray] = None

class SimpleVectorStore:
    """Simple in-memory vector store (use ChromaDB/FAISS in production)."""
    def __init__(self):
        self.documents: List[Document] = []
    
    def add(self, docs: List[Document]):
        self.documents.extend(docs)
    
    def search(self, query_emb: np.ndarray, k: int = 3) -> List[Dict]:
        if not self.documents:
            return []
        
        results = []
        for doc in self.documents:
            if doc.embedding is not None:
                score = float(np.dot(query_emb, doc.embedding) /
                             (np.linalg.norm(query_emb) * np.linalg.norm(doc.embedding) + 1e-8))
                results.append({"doc": doc, "score": score})
        
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:k]

def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks."""
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]

def mock_embed(text: str, dim: int = 16) -> np.ndarray:
    """Mock embedder — replace with real API in production."""
    np.random.seed(abs(hash(text)) % 2**31)
    return np.random.randn(dim).astype(np.float32)

# ── INDEXING PHASE ──
print("=== INDEXING PHASE ===")

raw_docs = {
    "python_basics.txt": """Python is a high-level, interpreted programming language.
    It supports multiple paradigms including OOP and functional programming.
    Python uses dynamic typing and automatic memory management.
    The Python ecosystem includes NumPy, Pandas, PyTorch, and many other libraries.
    Python was created by Guido van Rossum and first released in 1991.""",
    
    "rag_guide.txt": """RAG stands for Retrieval-Augmented Generation.
    It combines a retrieval system with a language model.
    The retrieval system finds relevant documents using semantic search.
    Retrieved documents are injected into the LLM's context window.
    RAG allows LLMs to access up-to-date or private information without fine-tuning.
    Popular RAG frameworks include LangChain, LlamaIndex, and Haystack.""",
}

store = SimpleVectorStore()
all_chunks = []

for filename, text in raw_docs.items():
    chunks = chunk_text(text, chunk_size=120, overlap=20)
    for i, chunk in enumerate(chunks):
        doc = Document(text=chunk, metadata={"source": filename, "chunk": i})
        doc.embedding = mock_embed(chunk)
        all_chunks.append(doc)
        
store.add(all_chunks)
print(f"Indexed {len(all_chunks)} chunks from {len(raw_docs)} documents")

In [ ]:
import numpy as np

# Continue from previous cell (simplified standalone version)
def mock_embed(text, dim=16):
    np.random.seed(abs(hash(text)) % 2**31)
    return np.random.randn(dim).astype(np.float32)

# ── RETRIEVAL PHASE ──
print("=== RETRIEVAL PHASE ===")

corpus = [
    ("Python is a high-level interpreted language", {"source": "python.txt"}),
    ("RAG combines retrieval with language generation", {"source": "rag.txt"}),
    ("NumPy provides fast array operations for Python", {"source": "numpy.txt"}),
    ("LangChain is a framework for LLM applications", {"source": "langchain.txt"}),
    ("Vector databases store embeddings for fast search", {"source": "vecdb.txt"}),
]

query = "What is RAG and how does it work?"
query_emb = mock_embed(query)

# Score all documents
scores = []
for text, meta in corpus:
    doc_emb = mock_embed(text)
    score = float(np.dot(query_emb, doc_emb) / 
                  (np.linalg.norm(query_emb) * np.linalg.norm(doc_emb) + 1e-8))
    scores.append((score, text, meta))

scores.sort(reverse=True)
top_k = scores[:3]

print(f"Query: '{query}'\n")
print("Retrieved documents:")
for score, text, meta in top_k:
    print(f"  [{score:.3f}] ({meta['source']}) {text}")

# ── GENERATION PHASE ──
print("\n=== GENERATION PHASE ===")

context = "\n".join([f"[{meta['source']}]: {text}" for _, text, meta in top_k])
prompt = f"""Answer the question using ONLY the provided context. If unsure, say so.

Context:
{context}

Question: {query}

Answer:"""

print("Prompt sent to LLM:")
print(prompt)
print("\n[LLM would generate a grounded answer here]")
print("\n✅ RAG Pipeline complete!")

## 🔗 What's Next?
[08 — Agents & Tools →](08_agents_and_tools.ipynb)